# Timme et al. (2016) on CRCNS hc-8: avalanches, exponents, branching ratio and randomizations

Paper: Timme, Marshall, Bennett, Ripp, Lautzenhiser & Beggs (2016), "Criticality Maximizes
Complexity in Neural Tissue", Front. Physiol. 7:425, doi:10.3389/fphys.2016.00425. The claims file
`claims/crcns-hc-8/10.3389-fphys.2016.00425.yaml` was written from the version of record
(PMC5037237) with its supplement, and checked by a second reader.

Dataset: CRCNS hc-8, doi:10.6080/K0PC308P, read from the public bucket
`crcnsarchive/hc-8/live/Data.zip` (435 files `Data/CultureXDIVY.mat`). Version: CRCNS data sets
are not versioned, so the version is `unversioned` (docs/archives.md). The zip read is the one
whose MD5 the data set's `checksums.md5` gives as `bd4e2a8685df3e55e1a3c04fbe09d9ce`; the data
description is version 0.9 (Sept. 28, 2016). This is the only published copy (the bucket's
`hc-8/orig/` holds a draft description, no data), and the paper names no version, so it is the
one pinned.

This notebook covers the culture claims that do not need the complexity. The complexity claims are
in `complexity_culture.ipynb` and the model claims in `models.ipynb`. The measures port the NCC
MATLAB toolbox the paper cites (see `hc8.py`).

| claim | here |
|---|---|
| culture-counts | tested |
| fig2d-network-isi | tested |
| fig2e-avalanche-count | tested |
| fig8-subsampling-exponents | tested |
| fig8-exponents-match-literature | blocked: the reference values of the three cited papers were not retrieved in this run |
| fig8-sigmanuz-methods-agree | tested |
| eq4-exponent-relation | tested |
| eq4-relation-randomized | tested |
| fig9-branching-ratio | tested |
| fig10b-susceptibility-randomization | tested |
| fig10c-curvature-randomization | tested |
| fig11ac-fit-range-randomization | tested |
| fig11bd-exponents-randomization | tested |

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
import time, resource, collections
import multiprocessing as mp
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import hc8

SEED = 20161
# HC8_SUBSET is used only while developing (a few recordings); the record runs all of them
SUBSET = int(os.environ.get("HC8_SUBSET", "0"))
names = hc8.recording_names()
idx = list(range(len(names))) if not SUBSET else list(range(0, len(names), len(names) // SUBSET))[:SUBSET]
t0 = time.time()
with mp.get_context("spawn").Pool(4) as pool:
    recs = list(pool.imap(hc8.culture_record, [(i, names[i], SEED) for i in idx], chunksize=1))
print(f"{len(recs)} recordings analysed in {(time.time() - t0) / 60:.1f} min; "
      f"peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
base = pd.DataFrame([{k: v for k, v in r.items() if k not in ("measures", "rates", "slopes")} for r in recs])
KINDS = ["real"] + hc8.RANDOMIZATIONS
meas = {k: pd.DataFrame([r["measures"][k] for r in recs], index=base.index) for k in KINDS}
rates = np.concatenate([r["rates"] for r in recs])

## Data check

The files hold one data stream, spike times per neuron, with no trial table or annotations, so
there is no second clock to align. The checks are of the description against the values.

In [ ]:
print("recordings:", len(base), "| cultures:", base.culture.nunique(),
      "| recordings per culture: min", base.groupby("culture").size().min(), "max", base.groupby("culture").size().max())
print("DIV range:", base["div"].min(), "to", base["div"].max())
print("neurons (data.nNeurons): total", base.n_neurons.sum(), "| min", base.n_neurons.min(),
      "max", base.n_neurons.max(), "mean", round(base.n_neurons.mean(), 1))
print("length(data.spikes) == nNeurons in every file:", bool((base.n_trains == base.n_neurons).all()))
print("data.binsize values (ms):", sorted(base.binsize.unique()), "| datatype:", base.datatype.unique(),
      "| expsys:", base.expsys.unique())
print("length(data.channel) == nNeurons:", bool((base.n_channel == base.n_neurons).all()))
dur_min = base.nbins_file * hc8.BIN_MS / 60e3
print(f"data.nbins as minutes: {dur_min.min():.2f} to {dur_min.max():.2f} (description: about 1 hour)")
print(f"last spike time: {base.t_max_ms.min() / 60e3:.2f} to {base.t_max_ms.max() / 60e3:.2f} min")
print("recordings with spike times beyond data.nbins:", int((base.n_beyond_nbins > 0).sum()),
      f"({int(base.n_beyond_nbins.sum())} spikes)")
print("every spike train sorted:", bool(base.all_sorted.all()),
      "| recordings with repeated spike times within a neuron:", int((base.n_dup_spikes > 0).sum()),
      f"({int(base.n_dup_spikes.sum())} repeats)")
print(f"shortest interval within a neuron: {base.min_isi_ms.min():.2f} ms; shortest non-zero: {base.min_pos_isi_ms.min():.2f} ms "
      f"(median over recordings {base.min_pos_isi_ms.median():.2f} ms)")
print("data.wave_ClusParams.ref values:", sorted(base.wave_clus_ref.unique()), "| sr values:", sorted(base.wave_clus_sr.unique()),
      "(Methods 2.2 say the 1.5 ms dead time was removed)")
print("identical recordings (same spike trains):", int(base.train_hash.duplicated().sum()),
      "| repeated recordingID:", int(base.recordingID.duplicated().sum()))
print("neurons with no spike in the first 59 min (dropped):", int(base.n_silent.sum()))
print(f"firing rate over the first 59 min: {rates.min():.4f} to {rates.max():.2f} Hz, "
      f"mean {rates.mean():.2f} Hz (description: average 1.9 Hz)")
print(f"mean network ISI: {base.isi_ms.min():.2f} to {base.isi_ms.max():.2f} ms; "
      f"rebinning factor {base.factor.min()} to {base.factor.max()} bins of 0.05 ms")
print("randomizations keep each neuron's spike count:",
      {k: bool(base[f"{k}_counts_kept"].all()) for k in hc8.RANDOMIZATIONS})
print("swap and shuffle leave A(t) unchanged:",
      bool(base.swap_act_identical.all()), bool(base.shuffle_act_identical.all()))

In [ ]:
# Exclusion for too few avalanches (claims file, common detail 2). Reading (a): the size and the
# duration distributions each keep at least one value after the cuts of Methods 2.7, and at least
# two durations enter the shape collapse; reading (b): all recordings.
r = meas["real"]
incl_a = (r.n_size_cut_values >= 1) & (r.n_dur_cut_values >= 1) & (r.n_profiles >= 2)
fit_ok = r.tau.notna() & r.alpha.notna() & r.sgd.notna() & r.sc_exp.notna()
print("recordings kept under reading (a):", int(incl_a.sum()), "of", len(r), "| excluded:", int((~incl_a).sum()))
print("recordings with every full-system fit available:", int(fit_ok.sum()))
print("excluded recordings:", list(base.name[~incl_a].str.replace("Data/", "")))
print("avalanches in the excluded recordings:", list(r.n_av[~incl_a]))

## culture-counts

> The culture data hold 435 recordings and 39,529 spike-sorted neurons in total; most recordings
> yield about 100 neurons, and firing rates range from roughly 0.05 to 20 Hz.

In [ ]:
nn = base.n_neurons.values
mode_nn, k_nn = hc8.terrell_scott_mode(nn)
print("number of recordings:", len(base), "(expected 435, tolerance 0)")
print("total neurons:", int(nn.sum()), "(expected 39529, tolerance 0)")
print(f"neurons per recording: (a) median {np.median(nn):.0f}; (b) Terrell-Scott mode {mode_nn:.1f} "
      f"({k_nn} bins); expected 100 +/- 10; range {nn.min()} to {nn.max()}")
p5, p95 = np.percentile(rates, [5, 95])
print(f"firing rates over {len(rates)} neurons: 5th percentile {p5:.5f} Hz, 95th {p95:.2f} Hz "
      f"(criterion: within 0.045 to 22 Hz); min {rates.min():.4f}, max {rates.max():.2f}")
pos = rates[rates > 0]
print(f"orders of magnitude spanned by non-zero rates: {np.log10(pos.max() / pos.min()):.2f} "
      f"(1st to 99th percentile: {np.log10(np.percentile(pos, 99) / np.percentile(pos, 1)):.2f})")
print("fraction of neurons below 0.05 Hz:", round(float(np.mean(rates < 0.05)), 3),
      "| above 20 Hz:", round(float(np.mean(rates > 20)), 4))
print("\nneurons per recording by culture (median):")
print(base.groupby("culture").n_neurons.median().astype(int).to_dict())
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(nn, bins=k_nn); ax[0].set_xlabel("neurons per recording")
ax[1].hist(np.log10(pos), bins=50); ax[1].set_xlabel("log10 firing rate (Hz)")
plt.tight_layout(); plt.show()

**Result: reproduced.** The data set holds 435 recordings and 39529 neurons, both exactly as stated. The median number of neurons per recording is 91 and the Terrell-Scott mode 93.3, both within 10% of 100. The 5th and 95th percentiles of the firing rate are 0.04548 Hz and 7.70 Hz, inside 0.045 to 22 Hz, and non-zero rates span 3.62 orders of magnitude.

## fig2d-network-isi

> Most recordings have a mean network-wide interspike interval of around 7 ms, though some are
> much higher or lower.

In [ ]:
isi = base.isi_ms.values
mode_isi, k_isi = hc8.terrell_scott_mode(isi)
print(f"mean network ISI per recording: (a) median {np.median(isi):.2f} ms; (b) Terrell-Scott mode "
      f"{mode_isi:.2f} ms ({k_isi} bins); expected 7 +/- 0.7 ms")
print(f"range {isi.min():.2f} to {isi.max():.2f} ms; 5th to 95th percentile "
      f"{np.percentile(isi, 5):.2f} to {np.percentile(isi, 95):.2f} ms")
print("by culture (median ms):", base.groupby("culture").isi_ms.median().round(1).to_dict())
plt.figure(figsize=(4, 3)); plt.hist(isi, bins=k_isi); plt.xlabel("mean network ISI (ms)"); plt.show()

**Result: reproduced.** The median mean network ISI is 6.67 ms, within 7 +/- 0.7 ms (the histogram mode, 8.50 ms, is outside). ISIs range from 0.91 to 152.78 ms, so some recordings are much higher or lower.

## fig2e-avalanche-count

> After rebinning to the network ISI, most recordings contain around 10,000 avalanches.

In [ ]:
nav = meas["real"].n_av.values
mode_nav, k_nav = hc8.terrell_scott_mode(nav)
print(f"avalanches per recording (first 59 min, rebinned to the network ISI): (a) median {np.median(nav):.0f}; "
      f"(b) Terrell-Scott mode {mode_nav:.0f} ({k_nav} bins); expected 10000 +/- 1000")
print(f"range {nav.min()} to {nav.max()}; 5th to 95th percentile {np.percentile(nav, 5):.0f} to {np.percentile(nav, 95):.0f}")
print("recordings with 9000 to 11000 avalanches:", int(np.sum((nav >= 9000) & (nav <= 11000))))
print("by culture (median):", meas["real"].groupby(base.culture).n_av.median().astype(int).to_dict())
plt.figure(figsize=(4, 3)); plt.hist(nav, bins=k_nav); plt.xlabel("avalanches per recording"); plt.show()

**Result: not reproduced.** After rebinning the first 59 min to the network ISI, the median number of avalanches per recording is 83289 and the histogram mode 81707, about eight times the stated 10,000; only 2 of 435 recordings have 9000 to 11000 avalanches.

## fig8-subsampling-exponents

> In the culture recordings, sub-sampling extrapolation gives lower values of tau and alpha with
> lower errors than the search algorithm on the full system, while 1/sigma nu z changes little
> between the two.

Full system: tau and alpha by the range search of `plparams.m` after the cuts, 1/sigma nu z from
size given duration over the fitted duration range and from the shape collapse. Sub-sampling
(Methods 2.10): 30 random sub-samples at sizes evenly spaced from 40% to 100% of the neurons
(layout reading a), MLE over all avalanches surviving the cuts, weighted least squares on 1/N with
weight N, intercept kept. Each sub-sample is rebinned to its own network ISI (main reading,
Methods 2.10: "These analyses were identical to those performed in the whole recorded system");
keeping the full system's bin is shown as a variant. Layout reading b (30 sub-samples at every
integer size) is run on four recordings below.

In [ ]:
def msd(x):
    x = np.asarray(x, dtype=float); x = x[np.isfinite(x)]
    return x.mean(), x.std(ddof=1), x.size

sel = incl_a
full = {"tau": meas["real"].tau[sel], "alpha": meas["real"].alpha[sel],
        "sgd": meas["real"].sgd[sel], "sc_exp": meas["real"].sc_exp[sel]}
rows = []
for tag in ("own", "fixed"):
    for k in ("tau", "alpha", "sgd", "sc_exp"):
        fm, fs, fn = msd(full[k]); sm, ss, sn = msd(base[f"ss_{tag}_{k}"][sel])
        rows.append(dict(rebinning=tag, exponent=k, full_mean=fm, full_sd=fs, full_n=fn,
                         ss_mean=sm, ss_sd=ss, ss_n=sn, rel_change=(sm - fm) / fm))
fig8 = pd.DataFrame(rows)
print(fig8.round(3).to_string())
for tag in ("own", "fixed"):
    t = fig8[fig8.rebinning == tag].set_index("exponent")
    print(f"\n[{tag} bin] tau lower: {t.ss_mean['tau'] < t.full_mean['tau']}, tau SD lower: {t.ss_sd['tau'] < t.full_sd['tau']}; "
          f"alpha lower: {t.ss_mean['alpha'] < t.full_mean['alpha']}, alpha SD lower: {t.ss_sd['alpha'] < t.full_sd['alpha']}; "
          f"|change| sgd {abs(t.rel_change['sgd']):.3f}, shape collapse {abs(t.rel_change['sc_exp']):.3f} (criterion < 0.10)")
print("\nall recordings with finite values (exclusion reading b):")
for k in ("tau", "alpha", "sgd", "sc_exp"):
    print(k, "full %.3f +/- %.3f (n=%d)" % msd(meas["real"][k]), "| sub-sampled (own bin) %.3f +/- %.3f (n=%d)" % msd(base[f"ss_own_{k}"]))
print("\nfull-system fits: size p >= 0.2 found in", int(meas["real"].tau.notna().sum()), "recordings; duration in",
      int(meas["real"].alpha.notna().sum()))
print("full-system tau range %.3f to %.3f; sub-sampled (own bin) %.3f to %.3f" % (
    np.nanmin(full["tau"]), np.nanmax(full["tau"]), np.nanmin(base.ss_own_tau[sel]), np.nanmax(base.ss_own_tau[sel])))
print("\nby culture, median tau full vs sub-sampled (own bin):")
print(pd.DataFrame({"full": meas["real"].tau[sel].groupby(base.culture[sel]).median(),
                    "ss": base.ss_own_tau[sel].groupby(base.culture[sel]).median()}).round(2).T.to_string())
fig, ax = plt.subplots(1, 4, figsize=(13, 3))
for a, k in zip(ax, ("tau", "alpha", "sgd", "sc_exp")):
    a.hist(full[k].dropna(), bins=30, alpha=.6, label="full"); a.hist(base[f"ss_own_{k}"][sel].dropna(), bins=30, alpha=.6, label="sub-sampled")
    a.set_xlabel(k)
ax[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
# Layout reading (b): 30 random sub-samples at every integer size from 40% to 100%, on one
# recording drawn with a fixed seed among those kept (the layout costs about 55 times layout (a);
# one recording is what the time limit allows), compared with layout (a) on the same recording.
rng_b = np.random.default_rng([SEED, 8])
i_b = int(rng_b.choice(np.flatnonzero(sel.values)))
n_b = int(base.n_neurons.iloc[i_b] - base.n_silent.iloc[i_b])
sizes_b = hc8.subsample_sizes(n_b, layout="all")
chunks = [(names[idx[i_b]], sizes_b[j::8], [SEED, idx[i_b], 9, j]) for j in range(8)]
t1 = time.time()
with mp.get_context("spawn").Pool(4) as pool:
    parts = pool.map(hc8.subsample_chunk, chunks)
vb = {k: sum((p_[k] for p_ in parts), []) for k in ("tau", "alpha", "sgd", "sc_exp", "n")}
lb = {k: hc8.wls_intercept(vb["n"], vb[k]) for k in ("tau", "alpha", "sgd", "sc_exp")}
print(f"layout (b) on {base.name.iloc[i_b]}: {len(sizes_b)} sub-samples in {(time.time() - t1) / 60:.1f} min")
print(pd.DataFrame({"layout a": [base[f"ss_own_{k}"].iloc[i_b] for k in lb], "layout b": list(lb.values()),
                    "full system": [meas["real"][k].iloc[i_b] for k in lb]}, index=list(lb)).round(3).to_string())

**Result: reproduced.** Sub-sampling lowers the mean tau from 2.491 to 2.106 and its SD from 0.623 to 0.407, and the mean alpha from 2.928 to 2.730 and its SD from 0.648 to 0.579. The mean 1/sigma nu z changes by 3.6% (size given duration) and 1.6% (shape collapse). With the full system's bin kept for the sub-samples the same holds (tau 1.745, alpha 2.356, changes of 1.9% and 3.4%). On the one recording run with layout (b), the extrapolated values differ from layout (a) by at most 0.044.

## fig8-exponents-match-literature

> The culture critical exponents from the sub-sampling method agree better with previously reported
> values for neural avalanches (Mazzoni et al. 2007, Pasquale et al. 2008, Friedman et al. 2012)
> than those from the search algorithm on the full system.

The reference values are in the three cited papers. They were not retrieved in this run, so the
cell below holds none and the claim is blocked; the full-system and sub-sampled means it would
compare are printed under fig8-subsampling-exponents.

In [ ]:
# REFERENCE VALUES (outside inputs; see the attempt record's `inputs`)
REFS = {}

In [ ]:
t_own = fig8[fig8.rebinning == "own"].set_index("exponent")
rows = []
for paper, vals in REFS.items():
    for k, v in vals.items():
        rows.append(dict(paper=paper, exponent=k, reference=v, full_mean=t_own.full_mean[k], ss_mean=t_own.ss_mean[k],
                         d_full=abs(t_own.full_mean[k] - v), d_ss=abs(t_own.ss_mean[k] - v)))
lit = pd.DataFrame(rows)
if len(lit):
    lit["ss_closer"] = lit.d_ss < lit.d_full
    print(lit.round(3).to_string())
    for k in ("tau", "alpha"):
        s = lit[lit.exponent == k]
        print(f"{k}: sub-sampling closer for {int(s.ss_closer.sum())} of {len(s)} cited papers that report it")
else:
    print("no reference values")

## fig8-sigmanuz-methods-agree

> The values of 1/sigma nu z from the size given duration fits and from the shape collapse agree
> within error.

In [ ]:
for tag in ("own", "fixed"):
    t = fig8[fig8.rebinning == tag].set_index("exponent")
    for which, m, s in (("full system", "full_mean", "full_sd"), ("sub-sampling", "ss_mean", "ss_sd")):
        if which == "full system" and tag == "fixed":
            continue
        diff = abs(t[m]["sgd"] - t[m]["sc_exp"]); bound = max(t[s]["sgd"], t[s]["sc_exp"])
        print(f"{which} ({tag} bin): size given duration {t[m]['sgd']:.3f} +/- {t[s]['sgd']:.3f}, shape collapse "
              f"{t[m]['sc_exp']:.3f} +/- {t[s]['sc_exp']:.3f}; |difference| {diff:.3f} vs larger SD {bound:.3f}: agree {diff < bound}")
d = (meas["real"].sgd - meas["real"].sc_exp)[sel]
print(f"per recording (full system): median difference {d.median():.3f}, |difference| < 0.1 in {float((d.abs() < 0.1).mean()):.2f} of recordings")

**Result: reproduced.** On the full system the two estimates of 1/sigma nu z are 1.411 +/- 0.157 and 1.467 +/- 0.204, a difference of 0.056; with sub-sampling 1.360 +/- 0.226 and 1.490 +/- 0.337, a difference of 0.130. Both differences are smaller than the larger SD.

## eq4-exponent-relation

> The culture critical exponents satisfy the relation (alpha - 1)/(tau - 1) = 1/sigma nu z (Eq. 4)
> with both the search algorithm and the sub-sampling method.

In [ ]:
def relation(tm, ts, am, as_):
    f = (am - 1) / (tm - 1)
    sf = np.sqrt((as_ / (tm - 1)) ** 2 + ((am - 1) * ts / (tm - 1) ** 2) ** 2)
    return f, sf

def eq4_table(tbl, m, s):
    f, sf = relation(tbl[m]["tau"], tbl[s]["tau"], tbl[m]["alpha"], tbl[s]["alpha"])
    out = {"ratio": f, "sd": sf, "sgd": tbl[m]["sgd"], "sc": tbl[m]["sc_exp"],
           "holds_sgd": abs(f - tbl[m]["sgd"]) <= sf, "holds_sc": abs(f - tbl[m]["sc_exp"]) <= sf}
    out["holds"] = out["holds_sgd"] or out["holds_sc"]
    return out

for tag in ("own", "fixed"):
    t = fig8[fig8.rebinning == tag].set_index("exponent")
    for which, m, s in (("search algorithm (full)", "full_mean", "full_sd"), ("sub-sampling", "ss_mean", "ss_sd")):
        if which.startswith("search") and tag == "fixed":
            continue
        e = eq4_table(t, m, s)
        print(f"{which} [{tag} bin]: (alpha-1)/(tau-1) = {e['ratio']:.3f} +/- {e['sd']:.3f}; 1/sigma nu z sgd {e['sgd']:.3f}, "
              f"shape collapse {e['sc']:.3f}; within one SD: sgd {e['holds_sgd']}, shape collapse {e['holds_sc']}")
rr = meas["real"][sel]
pr = (rr.alpha - 1) / (rr.tau - 1)
print(f"\nper recording, full system: median (alpha-1)/(tau-1) {pr.median():.3f}, median sgd {rr.sgd.median():.3f}; "
      f"|ratio - sgd| < 0.1 in {float(((pr - rr.sgd).abs() < 0.1).mean()):.2f} of recordings, < 0.2 in {float(((pr - rr.sgd).abs() < 0.2).mean()):.2f}")
for tag in ("own",):
    ps = (base[f"ss_{tag}_alpha"] - 1) / (base[f"ss_{tag}_tau"] - 1)
    print(f"per recording, sub-sampling: median ratio {ps[sel].median():.3f}, median sgd {base[f'ss_{tag}_sgd'][sel].median():.3f}")

**Result: reproduced.** From the mean exponents, (alpha - 1)/(tau - 1) is 1.293 +/- 0.694 for the search algorithm and 1.565 +/- 0.778 for sub-sampling, within one SD of both estimates of 1/sigma nu z (1.411 and 1.467; 1.360 and 1.490). The propagated SDs are large, so this test is weak: per recording, the ratio is within 0.1 of the size given duration exponent in 0.21 of recordings.

## eq4-relation-randomized

> Randomized culture data also satisfy the critical exponent relation, with different exponent values.

Full-system exponents of each randomization that changes the avalanches (swap and shuffling keep
A(t), so their exponents equal the real data's and cannot differ).

In [ ]:
real_m = meas["real"][sel]
rows = []
for k in ["jitter1", "jitter10", "jitter100", "wrap", "poisson"]:
    m_ = meas[k][sel]
    tm, ts, _ = msd(m_.tau); am, as_, _ = msd(m_.alpha); gm, gs, _ = msd(m_.sgd); cm, cs_, _ = msd(m_.sc_exp)
    f, sf = relation(tm, ts, am, as_)
    holds = (abs(f - gm) <= sf) or (abs(f - cm) <= sf)
    dt = abs(tm - real_m.tau.mean()) > real_m.tau.std(ddof=1)
    da = abs(am - real_m.alpha.mean()) > real_m.alpha.std(ddof=1)
    rows.append(dict(method=k, tau=tm, tau_sd=ts, alpha=am, alpha_sd=as_, ratio=f, ratio_sd=sf, sgd=gm, sc=cm,
                     relation_holds=holds, tau_differs=dt, alpha_differs=da))
eqr = pd.DataFrame(rows)
print(eqr.round(3).to_string())
print(f"\nreal data: tau {real_m.tau.mean():.3f} +/- {real_m.tau.std(ddof=1):.3f}, alpha {real_m.alpha.mean():.3f} +/- {real_m.alpha.std(ddof=1):.3f}")
ok = eqr[eqr.relation_holds & (eqr.tau_differs | eqr.alpha_differs)]
print("methods where the relation holds and tau or alpha differs by more than one SD:", list(ok.method))

**Result: reproduced.** The relation holds within one SD for all five randomizations that change the avalanches, and for wrap and Poisson both tau (3.262, 4.287) and alpha (3.659, 4.294) differ from the real data's (2.491 +/- 0.623, 2.928 +/- 0.648) by more than one SD.

## fig9-branching-ratio

> With the sub-sampling-corrected method, most culture recordings have branching ratios slightly
> below 1, while the basic method gives ratios substantially below 1 and more varied.

In [ ]:
bn, bb = base.B_new.values, base.B_basic.values
print(f"B_new: median {np.nanmedian(bn):.3f}, mean {np.nanmean(bn):.3f}, SD {np.nanstd(bn, ddof=1):.3f}, range {np.nanmin(bn):.3f} to {np.nanmax(bn):.3f}, "
      f"failed fits {int(np.isnan(bn).sum())}")
print(f"B_basic (Eq. 11): median {np.median(bb):.3f}, mean {bb.mean():.3f}, SD {bb.std(ddof=1):.3f}, range {bb.min():.3f} to {bb.max():.3f}")
for w in (0.05, 0.1):
    in_band = np.mean((bn >= 1 - w) & (bn < 1)); below = np.mean(bb < 1 - w)
    print(f"reading band {w}: B_new in [{1 - w}, 1) for {in_band:.3f} of recordings; B_basic below {1 - w} for {below:.3f}; "
          f"SD B_basic > SD B_new: {np.std(bb, ddof=1) > np.nanstd(bn, ddof=1)}")
print("B_new >= 1 in", int(np.sum(bn >= 1)), "recordings")
print(f"variants: brsimple (slope of A(t+1) on A(t), A(t) <= 10) median {base.B_simple.median():.3f}, SD {base.B_simple.std():.3f}; "
      f"avprops per-avalanche ratio median {base.B_avalanche.median():.3f}, SD {base.B_avalanche.std():.3f}")
print("\nby culture, fraction of recordings with B_new in [0.9, 1):")
print(base.assign(b=(bn >= 0.9) & (bn < 1)).groupby("culture").b.mean().round(2).to_dict())
fig, ax = plt.subplots(figsize=(4, 3)); ax.hist(bb, bins=40, alpha=.6, label="basic"); ax.hist(bn[np.isfinite(bn)], bins=40, alpha=.6, label="new")
ax.legend(); ax.set_xlabel("branching ratio"); plt.show()

**Result: reproduced.** B_new has median 0.973 and SD 0.064; 0.736 of recordings lie in [0.95, 1) and 0.874 in [0.9, 1). B_basic has median 0.798 and SD 0.088, below 0.95 in 0.947 of recordings. Both readings of the band satisfy the criterion; 16 recordings have B_new of 1 or more.

## Randomization tests (Figures 10B, 10C and 11)

The Figure 10 and 11 legends give a rank sum test of the real data against each randomization
across recordings, with false discovery rate control. Readings of the correction (claims file,
common detail 3): Benjamini-Hochberg (BH) or Benjamini-Yekutieli (BY), over the seven comparisons
of one panel; the family of all panels of the figure is shown as a variant. Recordings kept under
exclusion reading (a); reading (b), all recordings with finite values, is shown too.

In [ ]:
def panel(col, kinds=hc8.RANDOMIZATIONS, rows=sel):
    out = []
    x = meas["real"][col][rows].dropna()
    for k in kinds:
        y = meas[k][col][rows].dropna()
        p = stats.mannwhitneyu(x, y, alternative="two-sided").pvalue
        out.append(dict(randomization=k, real_median=x.median(), rand_median=y.median(),
                        change=y.median() - x.median(), p=p, n_real=len(x), n_rand=len(y)))
    t = pd.DataFrame(out).set_index("randomization")
    t["p_bh"] = hc8.fdr(t.p.values, "bh"); t["p_by"] = hc8.fdr(t.p.values, "by")
    return t

panels = {}
for col in ("chi_a", "chi_b", "abs_curv", "curv", "s_range", "d_range", "tau", "alpha"):
    panels[col] = panel(col)
fam11 = pd.concat([panels[c] for c in ("s_range", "tau", "d_range", "alpha")], keys=["s_range", "tau", "d_range", "alpha"])
fam11["p_bh_fig"] = hc8.fdr(fam11.p.values, "bh"); fam11["p_by_fig"] = hc8.fdr(fam11.p.values, "by")
panels_b = {col: panel(col, rows=slice(None)) for col in ("chi_a", "chi_b", "abs_curv", "s_range", "d_range", "tau", "alpha")}
FIVE = ["jitter1", "jitter10", "jitter100", "wrap", "poisson"]

## fig10b-susceptibility-randomization

> Culture susceptibility decreases under randomization, most under the strongest forms such as
> Poisson randomization.

In [ ]:
for col, lab in (("chi_a", "reading (a) Eq. 14 var(A/N)"), ("chi_b", "reading (b) N var(A/N)")):
    t = panels[col]
    print(lab); print(t.to_string(float_format=lambda v: f"{v:.3g}"))
    five = t.loc[FIVE]
    dec = five.change.sort_values()
    for c in ("p_bh", "p_by"):
        ok = bool(((five.change < 0) & (five[c] < 0.05)).all())
        print(f"  {c}: all five lower with corrected p < 0.05: {ok}")
    print("  largest decreases (median):", list(dec.index[:2]), "| Poisson among the two largest:", "poisson" in list(dec.index[:2]))
    print("  relative median change:", (five.change / five.real_median).round(3).to_dict())
    print("  exclusion reading (b):", {k: (round(v.change, 6), f"{v.p_bh:.2g}") for k, v in panels_b[col].loc[FIVE].iterrows()})
print("\nper culture, fraction of recordings with lower chi (reading b) under Poisson:",
      (meas["poisson"].chi_b < meas["real"].chi_b)[sel].groupby(base.culture[sel]).mean().round(2).to_dict())

## fig10c-curvature-randomization

> The culture shape collapse curvature decreases under randomization, so the collapsed avalanches
> become flatter.

In [ ]:
t = panels["abs_curv"]
print("absolute curvature (claims file, common detail 6)"); print(t.to_string(float_format=lambda v: f"{v:.3g}"))
five = t.loc[FIVE]
for c in ("p_bh", "p_by"):
    print(f"{c}: all five lower with corrected p < 0.05: {bool(((five.change < 0) & (five[c] < 0.05)).all())}; "
          f"lower and significant: {list(five.index[(five.change < 0) & (five[c] < 0.05)])}")
print("exclusion reading (b):", {k: (round(v.change, 4), f"{v.p_bh:.2g}") for k, v in panels_b["abs_curv"].loc[FIVE].iterrows()})
print("\nvariant, signed curvature (toolbox output):"); print(panels["curv"].loc[FIVE].to_string(float_format=lambda v: f"{v:.3g}"))
print(f"real data: signed curvature negative (arched) in {float((meas['real'].curv[sel] < 0).mean()):.2f} of recordings; "
      f"range {meas['real'].curv[sel].min():.3f} to {meas['real'].curv[sel].max():.3f}")
print("\nper culture, fraction of recordings with lower |curvature| under Poisson:",
      (meas["poisson"].abs_curv < meas["real"].abs_curv)[sel].groupby(base.culture[sel]).mean().round(2).to_dict())

In [ ]:
# Whole-figure family for Figure 10 panels B and C (panel A, complexity, is in complexity_culture.ipynb,
# which computes the family of all three panels)
fam10 = pd.concat([panels["chi_a"], panels["abs_curv"]], keys=["chi_a", "abs_curv"])
fam10["p_bh_fig"] = hc8.fdr(fam10.p.values, "bh"); fam10["p_by_fig"] = hc8.fdr(fam10.p.values, "by")
print(fam10.loc[(slice(None), FIVE), ["change", "p", "p_bh_fig", "p_by_fig"]].to_string(float_format=lambda v: f"{v:.3g}"))

**Result: partially reproduced.** The absolute shape collapse curvature falls from a median of 2.29 to 0.813 (10-bin jitter), 0.048 (100-bin jitter), 0.266 (wrap) and 0.0746 (Poisson), each with corrected p < 1e-37, but 1-bin jitter raises it to 2.43 (corrected p 0.391). Four of the five randomizations flatten the collapse.

**Result: reproduced.** With the Williams-Garcia form N var(A/N) and Benjamini-Hochberg, all five randomizations that change A(t) lower the median susceptibility with corrected p < 0.05 (largest corrected p 0.0231, for 1-bin jitter), and Poisson and wrap give the largest decreases (-72%). With Eq. 14 as printed, or with Benjamini-Yekutieli, 1-bin jitter is not significant (corrected p 0.0908 and 0.235 for Eq. 14).

## fig11ac-fit-range-randomization

> The size and duration power-law fit ranges decrease under strong randomization such as Poisson
> randomization, but increase slightly under 10-bin jittering.

In [ ]:
for col in ("s_range", "d_range"):
    t = panels[col]
    print(col, "(decades, log10 xmax/xmin)"); print(t.to_string(float_format=lambda v: f"{v:.3g}"))
    print(f"  Poisson lower: {t.change['poisson'] < 0}, p_bh {t.p_bh['poisson']:.3g}, p_by {t.p_by['poisson']:.3g}; "
          f"10-bin jitter higher median: {t.change['jitter10'] > 0} (change {t.change['jitter10']:.4f}, p_bh {t.p_bh['jitter10']:.3g})")
    print("  means real vs jitter10:", round(meas['real'][col][sel].mean(), 4), round(meas['jitter10'][col][sel].mean(), 4))
    print("  whole-figure family:", fam11.loc[col].loc[FIVE, ["change", "p_bh_fig", "p_by_fig"]].round(4).to_dict("index"))
print("\nper culture, fraction of recordings with a shorter size fit range under Poisson:",
      (meas["poisson"].s_range < meas["real"].s_range)[sel].groupby(base.culture[sel]).mean().round(2).to_dict())

**Result: reproduced.** Poisson randomization shortens the median size fit range from 0.694 to 0.248 decades and the duration fit range from 0.663 to 0.211 (corrected p about 4e-141), and 10-bin jitter lengthens them to 0.740 and 0.732 (corrected p 0.000135 and 5e-06). 100-bin jitter and wrap also shorten both ranges significantly, and 1-bin jitter does not change them (corrected p 0.455 and 0.398).

## fig11bd-exponents-randomization

> The size and duration power-law exponents increase under randomization, as randomization removes
> large or long avalanches and steepens the distributions.

In [ ]:
for col in ("tau", "alpha"):
    t = panels[col]
    print(col); print(t.to_string(float_format=lambda v: f"{v:.3g}"))
    five = t.loc[FIVE]
    for c in ("p_bh", "p_by"):
        up = list(five.index[(five.change > 0) & (five[c] < 0.05)])
        print(f"  {c}: higher with corrected p < 0.05 for {len(up)} of 5: {up}")
    print("  whole-figure family BH:", fam11.loc[col].loc[FIVE, "p_bh_fig"].round(4).to_dict())
for k in FIVE:
    m_ = meas[k][sel]
    print(f"{k}: median largest size kept by the cuts' fit xmax {m_.s_xmax.median():.0f} vs real {meas['real'].s_xmax[sel].median():.0f}; "
          f"avalanches {m_.n_av.median():.0f} vs {meas['real'].n_av[sel].median():.0f}")
print("\nper culture, fraction of recordings with higher tau under Poisson:",
      (meas["poisson"].tau > meas["real"].tau)[sel].groupby(base.culture[sel]).mean().round(2).to_dict())

**Result: partially reproduced.** Tau rises significantly under 10-bin jitter, 100-bin jitter, wrap and Poisson (4 of 5; median 2.50 to 4.53 under Poisson) and falls under 1-bin jitter (2.39, corrected p 0.0108). Alpha rises significantly under 100-bin jitter, wrap and Poisson (3 of 5); 1-bin and 10-bin jitter do not change it significantly (corrected p 0.107 and 0.401).

## Summary

| claim | outcome |
|---|---|
| culture-counts | reproduced |
| fig2d-network-isi | reproduced |
| fig2e-avalanche-count | not reproduced |
| fig8-subsampling-exponents | reproduced |
| fig8-exponents-match-literature | blocked (reference values not retrieved) |
| fig8-sigmanuz-methods-agree | reproduced |
| eq4-exponent-relation | reproduced |
| eq4-relation-randomized | reproduced |
| fig9-branching-ratio | reproduced |
| fig10b-susceptibility-randomization | reproduced |
| fig10c-curvature-randomization | partially reproduced |
| fig11ac-fit-range-randomization | reproduced |
| fig11bd-exponents-randomization | partially reproduced |

Differences from the paper's method: the NCC toolbox is ported to Python (`hc8.py`); jitter
proposes all moves at once instead of one spike at a time; layout reading (b) of the sub-sampling
was run on one recording only; no recording is excluded under reading (a) of the exclusion rule,
while the paper excluded some.

About the dataset: the counts of recordings (435) and neurons (39529) match the paper exactly, and
no recording is deposited twice. 47 recordings have spike times beyond `data.nbins` (64 spikes),
and 6 recordings have 21 repeated spike times within a neuron.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.